In [ ]:
# Standard library imports
import logging
import os
import ssl
import os
import importlib

# Third-party imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.nn.parameter import Parameter
from transformers import get_scheduler

# Local imports
from models import BaselineAgent
from utils import set_seed
from datasets import ObjectsDataset, CelebA, DSprites
from utils import load_dataset
from encoders import ProtoNetCNNEncoder, Dino

import datasets
import importlib
import encoders
import trainers

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# Configure SSL context to allow dataset downloads
ssl._create_default_https_context = ssl._create_unverified_context

logger.info("All necessary packages have been imported.")
logger.info("SSL context set to unverified to allow dataset downloads.")

## Config

In [ ]:
config_path = os.getenv('CONFIG', "config.py")

if config_path and os.path.exists(config_path):
    spec = importlib.util.spec_from_file_location("config", config_path)
    CFG = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(CFG)
else:
    print("No config file found.")

In [ ]:
from dataclasses import dataclass
from typing import Optional, Any
from datetime import datetime

@dataclass
class ExperimentConfig:
    """Configuration class for baseline reinforce experiment."""
    
    dataset: str = CFG.dataset

    # Model hyperparameters
    batch_size: int = 32
    vocab_size: int = CFG.vocab_size
    representation_dim: int = CFG.representation_dim
    input_dim: int = 40
    message_length: int = CFG.message_length
    
    # Training hyperparameters
    num_epochs: int = 100
    learning_rate: float = CFG.learning_rate_baseline
    weight_decay: float = 1e-5
    num_warmup_steps: int = 100
    
    # Training strategy parameters
    sampling_temperature: float = 1
    entropy_regularization_factor: float = CFG.entropy_regularization_factor
    contrastive_loss_temperature: float = CFG.contrastive_loss_temperature
    
    # Data split ratios
    train_split: float = 0.8
    val_split: float = 0.1
    test_split: float = 0.1
    
    # DataLoader parameters
    num_workers: int = 0
    drop_last: bool = True
    
    # Evaluation parameters
    number_of_candidates: int = CFG.number_of_candidates
    
    # System parameters
    seed: int = CFG.seed
    device: str = "auto"  # Will be set to 'cuda' or 'cpu' based on availability
    ckpt_dir: Optional[str] = None  # Will be generated from hyperparameters
    
    # Logging parameters
    log_level: str = "INFO"
    
    def __post_init__(self):
        """Post-initialization to set device and validate splits."""
        if self.device == "auto":
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        
        # Validate that splits sum to 1.0
        total_split = self.train_split + self.val_split + self.test_split
        if abs(total_split - 1.0) > 1e-6:
            raise ValueError(f"Data splits must sum to 1.0, got {total_split}")
        
        # Validate datasets
        valid_datasets = ['objects', 'shape', 'celeba', 'dsprites']
        if self.dataset not in valid_datasets:
            raise ValueError(f"dataset must be one of {valid_datasets}, got {self.dataset}")
        
        # Generate checkpoint directory from hyperparameters
        if self.ckpt_dir is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M")
            self.ckpt_dir = (
                f"runs/{timestamp}_bs{self.batch_size}_vocab{self.vocab_size}_"
                f"repr{self.representation_dim}_msglen{self.message_length}_"
                f"lr{self.learning_rate}_"
                f"temp{self.sampling_temperature}_ent{self.entropy_regularization_factor}_"
                f"contr{self.contrastive_loss_temperature}_"
                f"cand{self.number_of_candidates}_seed{self.seed}/checkpoints"
            )
    
    def to_dict(self) -> dict[str, Any]:
        """Convert config to dictionary."""
        return {
            'dataset': self.dataset,
            'batch_size': self.batch_size,
            'vocab_size': self.vocab_size,
            'representation_dim': self.representation_dim,
            'input_dim': self.input_dim,
            'message_length': self.message_length,
            'num_epochs': self.num_epochs,
            'learning_rate': self.learning_rate,
            'weight_decay': self.weight_decay,
            'num_warmup_steps': self.num_warmup_steps,
            'sampling_temperature': self.sampling_temperature,
            'entropy_regularization_factor': self.entropy_regularization_factor,
            'contrastive_loss_temperature': self.contrastive_loss_temperature,
            'train_split': self.train_split,
            'val_split': self.val_split,
            'test_split': self.test_split,
            'num_workers': self.num_workers,
            'drop_last': self.drop_last,
            'number_of_candidates': self.number_of_candidates,
            'seed': self.seed,
            'device': self.device,
            'ckpt_dir': self.ckpt_dir,
            'log_level': self.log_level
        }

# Create experiment configuration
config = ExperimentConfig()
logger.info(f"Experiment configuration created: {config}")


In [ ]:
# Use config parameters instead of hardcoded values
batch_size = config.batch_size
device = torch.device(config.device)  # config.device is guaranteed to be set in __post_init__
vocab_size = config.vocab_size
representation_dim = config.representation_dim
input_dim = config.input_dim
set_seed(config.seed)

In [ ]:
# Create dataset and split into train/validation/test sets
if config.dataset == 'objects':
    dataset = ObjectsDataset()
    train_dataset, val_dataset, test_dataset = random_split(dataset, [config.train_split, config.val_split, config.test_split])
    
    object_encoder_a = nn.Sequential(
        nn.Linear(config.input_dim, config.representation_dim)
    )
    object_encoder_b = nn.Sequential(
        nn.Linear(config.input_dim, config.representation_dim)
    )
    
elif config.dataset == 'shape':
    train_dataset = load_dataset("/home/shared/data/shape/train")
    val_dataset = load_dataset("/home/shared/data/shape/val")
    test_dataset = load_dataset("/home/shared/data/shape/test")
    
    object_encoder_a = ProtoNetCNNEncoder()
    object_encoder_b = ProtoNetCNNEncoder()
elif config.dataset == 'celeba':
    dataset = CelebA(n=50000)
    train_dataset, val_dataset, test_dataset = random_split(dataset, [config.train_split, config.val_split, config.test_split])
    
    object_encoder_a = encoders.Dino()
    object_encoder_b = encoders.Dino()
    
elif config.dataset == 'dsprites':
    dataset = DSprites(n=20000)
    train_dataset, val_dataset, test_dataset = random_split(dataset, [config.train_split, config.val_split, config.test_split])
    
    object_encoder_a = ProtoNetCNNEncoder(in_channels=1)
    object_encoder_b = ProtoNetCNNEncoder(in_channels=1)
    

train_loader = DataLoader(
    train_dataset, 
    batch_size=config.batch_size, 
    shuffle=True, 
    num_workers=config.num_workers, 
    drop_last=config.drop_last
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=config.batch_size, 
    shuffle=True, 
    num_workers=config.num_workers, 
    drop_last=config.drop_last
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=config.batch_size, 
    shuffle=False, 
    num_workers=config.num_workers
)

agent_a = BaselineAgent(
    input_dim=config.input_dim, 
    representation_dim=config.representation_dim, 
    vocab_size=config.vocab_size,
    object_encoder=object_encoder_a
).to(device)
agent_b = BaselineAgent(
    input_dim=config.input_dim, 
    representation_dim=config.representation_dim, 
    vocab_size=config.vocab_size,
    object_encoder=object_encoder_b
).to(device)

In [ ]:

optimizer = torch.optim.Adam(
    list[Parameter](agent_a.parameters())+list[Parameter](agent_b.parameters()), 
    lr=config.learning_rate, 
    weight_decay=config.weight_decay
)
num_training_steps = config.num_epochs * len(train_loader)

lr_scheduler = get_scheduler(
    'constant_with_warmup', 
    optimizer=optimizer, 
    num_warmup_steps=config.num_warmup_steps,
)

In [ ]:
from torch.utils.tensorboard import SummaryWriter

# Create tensorboard writer with the same directory structure as checkpoints
base_path = config.ckpt_dir.replace('/pretraining', '').replace('/checkpoints', '')
tensorboard_log_dir = f"{base_path}/tensorboard"
# tensorboard_log_dir = config.ckpt_dir.replace('checkpoints/', 'runs/') if config.ckpt_dir else 'runs/default'
writer = SummaryWriter(log_dir=tensorboard_log_dir)
logger.info(f"TensorBoard writer created at: {tensorboard_log_dir}")


In [ ]:
# config.contrastive_loss_temperature = 0.001

In [ ]:
# config.sampling_temperature = 1
# config.entropy_regularization_factor = 0.01 #0.2
# 1 --> too high
# 0.9 --> too low
# 0.1 --> too low

In [ ]:
import importlib
import trainers
import encoders
importlib.reload(encoders)
importlib.reload(trainers)

In [ ]:
from trainers import train_agents_baseline_reinforce

best_val_acc, best_model_state = train_agents_baseline_reinforce(
    agent_a=agent_a,
    agent_b=agent_b,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    device=device,
    num_epochs=config.num_epochs,
    message_length=config.message_length,
    sampling_temperature=config.sampling_temperature,
    entropy_regularization_factor=config.entropy_regularization_factor,
    contrastive_loss_temperature=config.contrastive_loss_temperature,
    ckpt_dir=config.ckpt_dir,
    tensorboard_writer=writer,
    logger=logging.getLogger(__name__)
)

In [ ]:
best_model_checkpoint_path = os.path.join(config.ckpt_dir, 'best_model.pth')
best_model_checkpoint = torch.load(best_model_checkpoint_path)
agent_a.load_state_dict(best_model_checkpoint['agent_a'])
agent_b.load_state_dict(best_model_checkpoint['agent_b'])

In [ ]:
from utils import evaluate_cross_communicate

test_accuracy = evaluate_cross_communicate(
    agent_a, agent_b, test_dataset, device, message_length=config.message_length, batch_size=config.number_of_candidates
)
logger.info(f"Test accuracy: {test_accuracy}")

In [ ]:
import json
import os

result_dir = tensorboard_log_dir.replace("tensorboard", "results")
os.makedirs(result_dir, exist_ok=True)

results = {
    "VQEL": False,
    "metrics": {
        "test_accuracy": round(test_accuracy, 3),
    },
    "config": {k: v for k, v in config.to_dict().items()}
}

with open(os.path.join(result_dir, "report.json"), "w") as file:
    json.dump(results, file, indent=4)